# Instructions

In this assignment, you will be working on **Smile Detection** using **YOLOv8** model, custom trained on 68 point facial landmarks dataset.

#### Task

You will have to complete the `smile_detector` function. In the function, you will receive an image and you have to write the logic for smile detection and return a boolean variable indicating whether the frame has a smiling person or not. 

We have provided the supporting code for keeping track of the smiling frames and we will use it to compare with the ground truth. If the percentage of overlap is high enough, you will obtain full marks.

Finally, we also write the output video in a avi file which you can download and see from the jupyter dashboard whether the smile is getting detected correctly.

You can see the output video to check whether the code detects smile or not and help to debug your code.

**You will have a total of 10 attempts.** This assignment carries 30 marks.

#### Some hints:

1. Find the lip and/or jaw coordinates using the facial landmarks.
1. For a person to be smiling, the ratio of width of the lip and the jaw width should be high.
1. Return `True` if a smile is detected, else return `False`.

#### Marks Distribution:
|Sections|Score|
|--------|-----|
|Implement `smile_detector` function| 30|


### Import the Ultralytics Library

In [5]:
# importing the necessary packages
import cv2
import os
import numpy as np
from ultralytics import YOLO  # Ensure you have the YOLO library installed
import matplotlib.pyplot as plt
%matplotlib inline

### YOLOv8 Model Loading:

In [2]:
# Load the YOLO model
MODEL_CKPT_PATH = os.path.join("../data", "yolov8n_face_kpts.pt")
model = YOLO(MODEL_CKPT_PATH)

## TODO : Complete the smile detector function

You need to apply the face detector and shape_predictor to the input image to get the landmarks. 

Then, Use the landmarks and come up with a logic so that the smile is detected for each frame.

Finally, override the variable **`isSmiling`** to True if smile is detected.

You can explain your logic in the next cell.

### Solution Logic
Write your logic here

In [3]:
def get_keypoints(image_path):
    # Predict keypoints using the YOLO model
    results = model(image_path, verbose=False)
    if len(results) == 0 or results[0].keypoints is None:
        return []
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    pred_keypoints = [tuple(map(float, kp)) for kp in keypoints]
    return pred_keypoints

### Visualize Facial Keypoints:

Lets, visualize the keypoints on a face and extracted using the YOLOv8 facial landmark model.

In [11]:
def visualize_keypoints(image_path, keypoints):
    # Load the image
    image = cv2.imread(image_path)
    
    # Convert float points to int
    keypoints = [tuple(map(int, kp)) for kp in keypoints]
    
    # Draw circles at the landmark points
    for point in keypoints:
        cv2.circle(image, point, 3, (0, 255, 0), -1)  # Green color with filled circle
    
    # Convert BGR image to RGB for matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Visualize using matplotlib
    plt.figure(figsize=(10,10))
    plt.imshow(image_rgb)
    plt.axis('off')  # Hide axes
    plt.show()

image_path = "../data/Alexandra Daddario Smile.jpeg"
    
# Get keypoints
keypoints = get_keypoints(image_path)

# Visualize keypoints
visualize_keypoints(image_path, keypoints)


In [4]:
def smile_detector(imDlib):
    keypoints = get_keypoints(imDlib)
    isSmiling = False

    if len(keypoints) >= 55:
        keypoints_arr = np.array(keypoints)
        lip_left = keypoints_arr[48]
        lip_right = keypoints_arr[54]
        jaw_left = keypoints_arr[0]
        jaw_right = keypoints_arr[16]
        lip_width = np.linalg.norm(lip_right - lip_left)
        jaw_width = np.linalg.norm(jaw_right - jaw_left)
        if jaw_width > 0:
            ratio = lip_width / jaw_width
            if ratio > 0.40:
                isSmiling = True
    # Return True if smile is detected
    return isSmiling

## Main function

This is the supporting function that does the video loading and saving part. 

It also calls the smile_detector function and keeps track of the smile_frames variable which is used in grading.

In [5]:
# Initializing video capture object.
capture = cv2.VideoCapture("../data/smile.mp4")
if(False == capture.isOpened()):
    print("[ERROR] Video not opened properly")    

# Create a VideoWriter object
smileDetectionOut = cv2.VideoWriter("smileDetectionOutput.avi",
                                   cv2.VideoWriter_fourcc('M','J','P','G'),
                                   15,(int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), 
                                       int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))))
    
frame_number = 0
smile_frames = []
while (True):
    # grab the next frame
    isGrabbed, frame = capture.read()
    if not isGrabbed:
        break
        
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_has_smile = smile_detector(image)
    if (True == frame_has_smile):
        cv2.putText(frame, "Smiling :)", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)
        smile_frames.append(frame_number)
    if frame_number % 50 == 0:
        print('\nProcessed {} frames'.format(frame_number))
        print("Smile detected in Frames: {}".format(smile_frames))
    # Write to VideoWriter
    smileDetectionOut.write(frame)
    
    frame_number += 1

capture.release()
smileDetectionOut.release()

In [6]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###
